# LISS-4 ClearNet: Cloud Removal Training & Evaluation
### Beyond the Clouds · BAH 2026

This notebook installs all dependencies, downloads the SEN12MS-CR simulation dataset, and runs the training/evaluation for LISS-4 ClearNet.

**Required Colab Runtime**: GPU (Runtime -> Change runtime type -> T4 GPU or A100)

## 1. Environment Setup & Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Install rasterio, scikit-image, and STAC client
!pip install -q rasterio scikit-image pystac-client requests tqdm wandb

# Create workspaces
import os
os.makedirs('/content/drive/MyDrive/LISS4_CloudRemoval/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/LISS4_CloudRemoval/results', exist_ok=True)
os.makedirs('/content/drive/MyDrive/LISS4_CloudRemoval/data', exist_ok=True)
os.makedirs('/content/src', exist_ok=True)

## 2. Generate Python Module Files locally in Colab

In [ ]:
# Let's write the STAC downloader client (complete version with search and download methods)
with open('/content/src/bhoonidhi_client.py', 'w') as f:
    f.write('''
import os
import requests
from pathlib import Path
from pystac_client import Client

class BhoonidhiClient:
    def __init__(self, username=None, password=None, base_url="https://bhoonidhi.nrsc.gov.in/bhoonidhi-api"):
        self.username = username or os.getenv("BHOO_USERNAME")
        self.password = password or os.getenv("BHOO_PASSWORD")
        self.base_url = base_url.rstrip('/')
        self.token = None
        self.headers = {}
        if self.username and self.password:
            self.authenticate()

    def authenticate(self):
        auth_url = f"{self.base_url}/auth/token"
        payload = {"userId": self.username, "password": self.password}
        try:
            response = requests.post(auth_url, json=payload, timeout=15)
            if response.status_code == 200:
                self.token = response.json().get("access_token")
                self.headers = {"Authorization": f"Bearer {self.token}"}
                print("\\u2705 Authenticated with Bhoonidhi STAC API.")
            else:
                print(f"\\u274c Auth failed: {response.status_code}")
        except Exception as e:
            print(f"\\u274c Connection failed: {e}")

    def search_scenes(self, collection, bbox, date_range, cloud_cover_gt=30):
        stac_url = f"{self.base_url}/stac"
        try:
            client = Client.open(stac_url, headers=self.headers)
            search = client.search(
                collections=[collection],
                bbox=bbox,
                datetime=date_range,
                query={"eo:cloud_cover": {"gt": cloud_cover_gt}}
            )
            items = list(search.get_items())
            print(f"Found {len(items)} matching items.")
            return items
        except Exception as e:
            print(f"Error querying STAC API: {e}")
            return []

    def download_asset(self, item, out_dir, asset_name="data"):
        out_path = Path(out_dir)
        out_path.mkdir(parents=True, exist_ok=True)
        asset = item.assets.get(asset_name)
        if not asset:
            tiff_assets = [k for k, v in item.assets.items() if v.media_type in [\'image/tiff\', \'image/geotiff\'] or k == \'data\']
            if tiff_assets:
                asset = item.assets.get(tiff_assets[0])
        if not asset:
            return None
        url = asset.href
        filename = out_path / f"{item.id}_{asset_name}.tif"
        try:
            with requests.get(url, headers=self.headers, stream=True, timeout=30) as r:
                r.raise_for_status()
                with open(filename, \'wb\') as f:
                    for chunk in r.iter_content(16384):
                        f.write(chunk)
            print(f"Downloaded {filename.name}")
            return filename
        except Exception as e:
            print(f"Failed to download: {e}")
            return None
''')
print("Created: bhoonidhi_client.py")

In [ ]:
# Write preprocessor
with open('/content/src/preprocess.py', 'w') as f:
    f.write('''
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from skimage.filters import threshold_otsu
from skimage.morphology import dilation, square
from pathlib import Path

class RemoteSensingPreprocessor:
    @staticmethod
    def warp_sar_to_optical(optical_path, sar_path, out_bands=2):
        with rasterio.open(optical_path) as opt_src:
            opt_transform = opt_src.transform
            opt_crs = opt_src.crs
            H, W = opt_src.height, opt_src.width
        sar_warped = np.zeros((out_bands, H, W), dtype=np.float32)
        with rasterio.open(sar_path) as sar_src:
            for b in range(min(out_bands, sar_src.count)):
                reproject(
                    source=rasterio.band(sar_src, b + 1),
                    destination=sar_warped[b],
                    src_transform=sar_src.transform,
                    src_crs=sar_src.crs,
                    dst_transform=opt_transform,
                    dst_crs=opt_crs,
                    resampling=Resampling.bilinear
                )
        return sar_warped

    @staticmethod
    def generate_nsci_mask(optical_path, dilation_kernel_size=5):
        with rasterio.open(optical_path) as src:
            green = src.read(1).astype(np.float32)
            nir = src.read(3).astype(np.float32)
        nsci = (green - nir) / (green + nir + 1e-8)
        try:
            thresh = threshold_otsu(nsci)
            mask = (nsci > thresh).astype(np.uint8)
        except ValueError:
            mask = (nsci > 0.0).astype(np.uint8)
        if dilation_kernel_size > 0:
            mask = dilation(mask, square(dilation_kernel_size))
        return mask, nsci
''')
print("Created: preprocess.py")

In [ ]:
# Write dataset loader
with open('/content/src/dataset.py', 'w') as f:
    f.write('''
import torch
import numpy as np
from torch.utils.data import Dataset
from pathlib import Path
import rasterio

class SEN12MS_LISS4_SimulationDataset(Dataset):
    LISS4_BAND_MAP = [2, 3, 7]
    def __init__(self, root_dir, split=\'train\', patch_size=256):
        self.root = Path(root_dir)
        self.patch_size = patch_size
        self.pairs = []
        if self.root.exists():
            for season_dir in self.root.iterdir():
                if not season_dir.is_dir():
                    continue
                s1_dir = season_dir / \'s1\'
                s2c_dir = season_dir / \'s2_cloudy\'
                s2cf_dir = season_dir / \'s2_cloud_free\'
                if s1_dir.exists() and s2c_dir.exists() and s2cf_dir.exists():
                    s1_files = sorted(s1_dir.glob(\'*.tif\'))
                    for s1_f in s1_files:
                        idx = s1_f.stem.split(\'_\')[-1]
                        s2c_f = s2c_dir / f"s2_{idx}.tif"
                        s2cf_f = s2cf_dir / f"s2_{idx}.tif"
                        if s2c_f.exists() and s2cf_f.exists():
                            self.pairs.append((s1_f, s2c_f, s2cf_f))
        n = len(self.pairs)
        if split == \'train\':
            self.pairs = self.pairs[:int(0.85 * n)]
        else:
            self.pairs = self.pairs[int(0.85 * n):]
        print(f"Loaded {len(self.pairs)} items for {split}.")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        s1_path, s2c_path, s2cf_path = self.pairs[idx]
        with rasterio.open(s1_path) as src: sar = src.read().astype(np.float32)
        with rasterio.open(s2c_path) as src: s2c = src.read().astype(np.float32)[self.LISS4_BAND_MAP]
        with rasterio.open(s2cf_path) as src: s2cf = src.read().astype(np.float32)[self.LISS4_BAND_MAP]
        
        sar = np.clip(sar / 10000.0, -1.0, 1.0)
        cloudy = np.clip(s2c / 3000.0, 0.0, 1.0)
        clear = np.clip(s2cf / 3000.0, 0.0, 1.0)
        
        H, W = cloudy.shape[1], cloudy.shape[2]
        if H > self.patch_size:
            t = np.random.randint(0, H - self.patch_size)
            l = np.random.randint(0, W - self.patch_size)
            sl = (slice(None), slice(t, t + self.patch_size), slice(l, l + self.patch_size))
            sar, cloudy, clear = sar[sl], cloudy[sl], clear[sl]
        return torch.tensor(sar), torch.tensor(cloudy), torch.tensor(clear)
''')
print("Created: dataset.py")

In [ ]:
# Write model
with open('/content/src/model.py', 'w') as f:
    f.write('''
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResNetBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels)
        )
    def forward(self, x): return F.relu(x + self.conv(x))

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, embed_dim, num_heads=8):
        super().__init__()
        self.embed_dim, self.num_heads = embed_dim, num_heads
        self.head_dim = embed_dim // num_heads
        self.q_proj = nn.Conv2d(embed_dim, embed_dim, kernel_size=1)
        self.k_proj = nn.Conv2d(embed_dim, embed_dim, kernel_size=1)
        self.v_proj = nn.Conv2d(embed_dim, embed_dim, kernel_size=1)
        self.out_proj = nn.Conv2d(embed_dim, embed_dim, kernel_size=1)

    def forward(self, opt, sar):
        B, C, H, W = opt.shape
        q = self.q_proj(opt).view(B, self.num_heads, self.head_dim, H*W).transpose(-2, -1)
        k = self.k_proj(sar).view(B, self.num_heads, self.head_dim, H*W)
        v = self.v_proj(sar).view(B, self.num_heads, self.head_dim, H*W).transpose(-2, -1)
        scores = torch.matmul(q, k) / (self.head_dim**0.5)
        out = torch.matmul(F.softmax(scores, dim=-1), v).transpose(-2, -1).contiguous().view(B, C, H, W)
        return opt + self.out_proj(out)

class LISS4ClearNet(nn.Module):
    def __init__(self, num_res_blocks=16, channel_dim=128):
        super().__init__()
        self.opt_conv = nn.Sequential(nn.Conv2d(3, channel_dim, kernel_size=3, padding=1), nn.BatchNorm2d(channel_dim), nn.ReLU(True), ResNetBlock(channel_dim))
        self.sar_conv = nn.Sequential(nn.Conv2d(2, channel_dim, kernel_size=3, padding=1), nn.BatchNorm2d(channel_dim), nn.ReLU(True), ResNetBlock(channel_dim))
        self.cross_attention = MultiHeadCrossAttention(channel_dim)
        self.res_loop = nn.Sequential(*[ResNetBlock(channel_dim) for _ in range(num_res_blocks)])
        self.out_head = nn.Sequential(nn.Conv2d(channel_dim, channel_dim//2, kernel_size=3, padding=1), nn.ReLU(True), nn.Conv2d(channel_dim//2, 3, kernel_size=3, padding=1), nn.Sigmoid())
    def forward(self, opt, sar):
        return self.out_head(self.res_loop(self.cross_attention(self.opt_conv(opt), self.sar_conv(sar))))
''')
print("Created: model.py")

In [ ]:
# Write training script
with open('/content/src/train.py', 'w') as f:
    f.write('''
import os, argparse, torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
from dataset import SEN12MS_LISS4_SimulationDataset
from model import LISS4ClearNet

def ndvi_loss(pred, target):
    ndvi_p = (pred[:,2] - pred[:,1]) / (pred[:,2] + pred[:,1] + 1e-8)
    ndvi_t = (target[:,2] - target[:,1]) / (target[:,2] + target[:,1] + 1e-8)
    return F.mse_loss(ndvi_p, ndvi_t)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_dir", type=str, required=True)
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--output_dir", type=str, default="/content/drive/MyDrive/LISS4_CloudRemoval/checkpoints")
    args = parser.parse_args()
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_ds = SEN12MS_LISS4_SimulationDataset(args.data_dir, split="train")
    train_dl = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, num_workers=2)
    
    model = LISS4ClearNet().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-4)
    
    for epoch in range(1, args.epochs + 1):
        model.train()
        loss_sum = 0
        for sar, cloudy, clear in train_dl:
            sar, cloudy, clear = sar.to(device), cloudy.to(device), clear.to(device)
            pred = model(cloudy, sar)
            loss = F.mse_loss(pred, clear) + 0.5 * ndvi_loss(pred, clear)
            opt.zero_grad(); loss.backward(); opt.step()
            loss_sum += loss.item()
        print(f"Epoch {epoch:2d}/{args.epochs} | Loss: {loss_sum/len(train_dl):.5f}")
        if epoch % 5 == 0:
            torch.save(model.state_dict(), os.path.join(args.output_dir, f"model_epoch_{epoch}.pt"))
''')
print("Created: train.py")

## 3. Download the Dataset Test Split (Subset) via Rsync
This split is around ~15 GB (manageable inside Colab disk space).

In [ ]:
# Download spring season split (~15 GB) using standard environment variable for password to avoid permission errors
!export RSYNC_PASSWORD=m1659251 && rsync -chavzP \
  rsync://m1659251@dataserv.ub.tum.de/m1659251/ROIs1158_spring/ \
  /content/drive/MyDrive/LISS4_CloudRemoval/data/ROIs1158_spring/

## 4. Run Training

In [ ]:
# Start training
!python /content/src/train.py \
  --data_dir /content/drive/MyDrive/LISS4_CloudRemoval/data/ \
  --epochs 50 \
  --batch_size 8 \
  --output_dir /content/drive/MyDrive/LISS4_CloudRemoval/checkpoints/